In [ ]:
import kagglehub
import pandas as pd
import os

os.makedirs("data", exist_ok=True)

path = kagglehub.dataset_download("wordsforthewise/lending-club")
print("Đã tải về:", path)

print("\nCác file trong dataset:")
for f in os.listdir(path):
    full_path = os.path.join(path, f)
    size_mb = os.path.getsize(full_path) / 1e6
    print(f"  {f} — {size_mb:.1f} MB")


In [ ]:
import pandas as pd

file_path = r"C:\Users\HP\.cache\kagglehub\datasets\wordsforthewise\lending-club\versions\3\accepted_2007_to_2018Q4.csv.gz"

df_sample = pd.read_csv(file_path, compression='gzip', nrows=1000, low_memory=False)

print(f"Số cột: {df_sample.shape[1]}")
print(f"\nDanh sách toàn bộ cột:")
print(df_sample.columns.tolist())

key_cols = ['loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade', 'issue_d', 
            'loan_status', 'purpose', 'annual_inc', 'dti', 'last_pymnt_d', 
            'total_pymnt', 'home_ownership', 'emp_length']
existing = [c for c in key_cols if c in df_sample.columns]
missing = [c for c in key_cols if c not in df_sample.columns]
print(f"\nCột cốt lõi CÓ trong dataset: {existing}")
print(f"Cột cốt lõi THIẾU (cần kiểm tra tên khác): {missing}")

print(f"\n--- Phân phối loan_status (trên mẫu 1000 dòng) ---")
print(df_sample['loan_status'].value_counts())


In [ ]:
import pandas as pd
import numpy as np

file_path = r"C:\Users\HP\.cache\kagglehub\datasets\wordsforthewise\lending-club\versions\3\accepted_2007_to_2018Q4.csv.gz"

usecols = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade',
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
    'issue_d', 'loan_status', 'purpose', 'dti', 'delinq_2yrs',
    'fico_range_low', 'fico_range_high', 'open_acc', 'pub_rec', 'revol_bal',
    'revol_util', 'total_acc', 'last_pymnt_d', 'last_credit_pull_d',
    'application_type', 'mort_acc', 'pub_rec_bankruptcies',
    'hardship_flag', 'hardship_type', 'hardship_status',
    'debt_settlement_flag', 'settlement_status'
]

df = pd.read_csv(file_path, compression='gzip', usecols=usecols, low_memory=False)
print(f"Đã load: {df.shape[0]:,} dòng, {df.shape[1]} cột")

df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['last_pymnt_d'] = pd.to_datetime(df['last_pymnt_d'], format='%b-%Y')
df['last_credit_pull_d'] = pd.to_datetime(df['last_credit_pull_d'], format='%b-%Y')

df['end_date'] = df['last_pymnt_d'].fillna(df['last_credit_pull_d'])

df['duration'] = ((df['end_date'].dt.year - df['issue_d'].dt.year) * 12 + 
                   (df['end_date'].dt.month - df['issue_d'].dt.month)).clip(lower=1)

def map_event(status):
    if status == 'Charged Off':
        return 1
    elif status == 'Fully Paid':
        return 2
    else:
        return 0

df['event_type'] = df['loan_status'].apply(map_event)

print(f"\n--- Phân phối event_type trên toàn bộ {df.shape[0]:,} khoản vay ---")
print(df['event_type'].value_counts())
print(f"\n--- Phân phối loan_status gốc ---")
print(df['loan_status'].value_counts())

print(f"\n--- Thống kê duration (tháng) ---")
print(df['duration'].describe())

print(f"\n--- Tỷ lệ tham gia Hardship Plan ---")
print(df['hardship_flag'].value_counts())
print(f"\n--- Tỷ lệ tham gia Debt Settlement ---")
print(df['debt_settlement_flag'].value_counts())


In [ ]:
print(f"Số dòng loan_status bị NaN: {df['loan_status'].isnull().sum():,}")
print(f"Tỷ lệ: {df['loan_status'].isnull().sum() / len(df):.4%}")

def map_event_fixed(status):
    if pd.isnull(status):
        return -1
    if 'Charged Off' in status:
        return 1
    elif 'Fully Paid' in status:
        return 2
    elif status == 'Default':
        return 1
    else:
        return 0

df['event_type'] = df['loan_status'].apply(map_event_fixed)

print("--- Phân phối event_type SAU khi sửa ---")
print(df['event_type'].value_counts())

n_missing = (df['event_type'] == -1).sum()
if n_missing > 0:
    print(f"\nSố dòng bị loại do thiếu loan_status: {n_missing:,} ({n_missing/len(df):.4%})")
    df = df[df['event_type'] != -1].copy()
    print(f"Số dòng còn lại sau khi loại: {len(df):,}")

df['term_months'] = df['term'].str.extract(r'(\d+)').astype(float)

over_term = df[df['duration'] > df['term_months'] + 2]
print(f"Số khoản vay có duration vượt kỳ hạn (term) hơn 2 tháng: {len(over_term):,} "
      f"({len(over_term)/len(df):.2%})")
print(over_term[['term_months', 'duration', 'loan_status']].describe())

raw_duration_check = ((df['end_date'].dt.year - df['issue_d'].dt.year) * 12 + 
                       (df['end_date'].dt.month - df['issue_d'].dt.month))
n_negative_or_zero = (raw_duration_check <= 0).sum()
print(f"\nSố dòng có duration gốc <= 0 trước khi clip (end_date <= issue_d): "
      f"{n_negative_or_zero:,} ({n_negative_or_zero/len(df):.2%})")


In [ ]:
before = len(df)

df_clean = df[
    (df['duration'] <= df['term_months'] + 2) &
    (raw_duration_check > 0)
].copy()

print(f"Trước khi lọc: {before:,} dòng")
print(f"Sau khi lọc: {len(df_clean):,} dòng")
print(f"Đã loại: {before - len(df_clean):,} dòng ({(before-len(df_clean))/before:.2%})")

print(f"\n--- Phân phối event_type cuối cùng (dữ liệu sạch) ---")
print(df_clean['event_type'].value_counts())
print(f"\n--- Duration sau khi làm sạch ---")
print(df_clean['duration'].describe())

df_clean.to_csv("data/lendingclub_clean.csv", index=False)
print("\nĐã lưu data/lendingclub_clean.csv")


In [ ]:
import pandas as pd
import numpy as np
from lifelines import AalenJohansenFitter, KaplanMeierFitter
import matplotlib.pyplot as plt

df = pd.read_csv("data/lendingclub_clean.csv", low_memory=False)

kmf_naive = KaplanMeierFitter()
event_charged_off_naive = (df['event_type'] == 1).astype(int)
kmf_naive.fit(durations=df['duration'], event_observed=event_charged_off_naive)
naive_risk_at_36m = 1 - kmf_naive.survival_function_at_times(36).values[0]
print(f"[SAI - KM thường] Ước lượng xác suất vỡ nợ tại tháng 36: {naive_risk_at_36m:.2%}")

ajf_default = AalenJohansenFitter()
ajf_default.fit(durations=df['duration'], event_observed=df['event_type'], event_of_interest=1)

cif_table = ajf_default.cumulative_density_
correct_risk_at_36m = cif_table.asof(36).values[0]

print(f"[ĐÚNG - Aalen-Johansen] Xác suất vỡ nợ tại tháng 36: {correct_risk_at_36m:.2%}")
print(f"\nMức độ lệch nếu dùng sai phương pháp: {naive_risk_at_36m - correct_risk_at_36m:.2%} điểm phần trăm")

ajf_paid = AalenJohansenFitter()
ajf_paid.fit(durations=df['duration'], event_observed=df['event_type'], event_of_interest=2)

fig, ax = plt.subplots(figsize=(9, 5))
ajf_default.plot(ax=ax, label='Charged Off (vỡ nợ)', color='#ff4b4b')
ajf_paid.plot(ax=ax, label='Fully Paid (trả hết sớm)', color='#00d4aa')
plt.title("Cumulative Incidence Function — 2 rủi ro cạnh tranh")
plt.xlabel("Tháng kể từ lúc giải ngân")
plt.ylabel("Xác suất tích lũy")
plt.legend()
plt.savefig("data/cif_competing_risks.png")
print("\nĐã lưu biểu đồ: data/cif_competing_risks.png")

print("\n--- Xác suất vỡ nợ tại tháng 36, theo Grade ---")
for grade in sorted(df['grade'].dropna().unique()):
    mask = df['grade'] == grade
    ajf_g = AalenJohansenFitter()
    ajf_g.fit(durations=df.loc[mask, 'duration'], 
              event_observed=df.loc[mask, 'event_type'], event_of_interest=1)
    risk = ajf_g.cumulative_density_.asof(36).values[0]
    print(f"Grade {grade}: {risk:.2%} (n={mask.sum():,})")


In [ ]:
import pandas as pd
from lifelines import CoxPHFitter

cox_features_v2 = ['loan_amnt', 'grade', 'term', 'annual_inc', 'dti',
                    'fico_range_low', 'home_ownership', 'purpose', 'verification_status',
                    'emp_length', 'delinq_2yrs', 'open_acc', 'pub_rec', 'revol_util',
                    'application_type']

df = pd.read_csv("data/lendingclub_clean.csv", low_memory=False)
df['emp_length'] = df['emp_length'].fillna('Missing')
df['home_ownership'] = df['home_ownership'].replace({'NONE': 'OTHER'})

cox_df = df[['duration', 'event_type'] + cox_features_v2].copy()
cox_df['revol_util'] = cox_df['revol_util'].fillna(cox_df['revol_util'].median())
cox_df['dti'] = cox_df['dti'].fillna(cox_df['dti'].median())
cox_df = cox_df.dropna(subset=[c for c in cox_features_v2 if c != 'emp_length'])

cox_df = pd.get_dummies(cox_df, columns=['grade', 'term', 'home_ownership', 'purpose',
                                           'verification_status', 'emp_length',
                                           'application_type'], drop_first=True)

cox_df_default = cox_df.copy()
cox_df_default['event_default'] = (cox_df_default['event_type'] == 1).astype(int)
cox_df_default = cox_df_default.drop(columns=['event_type'])

sample_default = cox_df_default.sample(n=200_000, random_state=42)

cph_default_v2 = CoxPHFitter(penalizer=0.01)
cph_default_v2.fit(sample_default, duration_col='duration', event_col='event_default')

print("=== Cause-Specific Cox PH — CHARGED OFF (v2: đã loại int_rate) ===")
print(f"Concordance: {cph_default_v2.concordance_index_:.4f}")
cph_default_v2.print_summary(columns=['coef', 'exp(coef)', 'p'])

cph_default_v2.save("data/cox_default_model.pkl") if hasattr(cph_default_v2, 'save') else None


In [ ]:
import pandas as pd
from lifelines import CoxPHFitter

df = pd.read_csv("data/lendingclub_clean.csv", low_memory=False)
df['emp_length'] = df['emp_length'].fillna('Missing')
df['home_ownership'] = df['home_ownership'].replace({'NONE': 'OTHER'})

cox_features_v2 = ['loan_amnt', 'grade', 'term', 'annual_inc', 'dti',
                    'fico_range_low', 'home_ownership', 'purpose', 'verification_status',
                    'emp_length', 'delinq_2yrs', 'open_acc', 'pub_rec', 'revol_util',
                    'application_type']

cox_df = df[['duration', 'event_type'] + cox_features_v2].copy()
cox_df['revol_util'] = cox_df['revol_util'].fillna(cox_df['revol_util'].median())
cox_df['dti'] = cox_df['dti'].fillna(cox_df['dti'].median())
cox_df = cox_df.dropna(subset=[c for c in cox_features_v2 if c != 'emp_length'])

cox_df = pd.get_dummies(cox_df, columns=['grade', 'term', 'home_ownership', 'purpose',
                                           'verification_status', 'emp_length',
                                           'application_type'], drop_first=True)

cox_df_paid = cox_df.copy()
cox_df_paid['event_paid'] = (cox_df_paid['event_type'] == 2).astype(int)
cox_df_paid = cox_df_paid.drop(columns=['event_type'])

sample_paid = cox_df_paid.sample(n=200_000, random_state=42)

cph_paid = CoxPHFitter(penalizer=0.01)
cph_paid.fit(sample_paid, duration_col='duration', event_col='event_paid')

print("=== Cause-Specific Cox PH — FULLY PAID (trả sớm) ===")
print(f"Concordance: {cph_paid.concordance_index_:.4f}")
cph_paid.print_summary(columns=['coef', 'exp(coef)', 'p'])
